# Authors

- David Robredo Manuel
- Duarte Novas Álvarez
- Rubén González Braña

## Imports

In [ ]:
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import visualkeras

from keras.callbacks import EarlyStopping

from keras.utils import to_categorical

In [ ]:
print("Available Physical Devices: ", tf.config.list_physical_devices())

## Functions

In [ ]:
def show_image(image):

    plt.imshow(image)
    plt.axis("off")
    plt.show()

In [ ]:
import copy
import cv2
import numpy as np

def canny_edge_dataset(dataset):
    canny_dataset = []
    for img_pre in dataset:
        try:
            img = copy.deepcopy(img_pre)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 150, 300)
            img[edges == 255] = (255, 0, 0)
            canny_dataset.append(img)
        except:
            print("Couldn't process image")

    return np.array(canny_dataset)


def sift_transform_dataset(dataset):
    sift_dataset = []
    for img_pre in dataset:
        try:
            img = img_pre.copy()
            sift = cv2.SIFT_create()
            keypoints = sift.detect(img, None)
            img_keypoints = cv2.drawKeypoints(img, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
            sift_dataset.append(img_keypoints)
        except:
            print("Couldn't process image")
    
    return np.array(sift_dataset)

## Preprocessing

#### LOADING DATA

We load the data and divide it into train and test. Also, we assert that there are the correct number of instances in each group.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar100.load_data(label_mode="coarse")
assert X_train.shape == (50000, 32, 32, 3)
assert X_test.shape == (10000, 32, 32, 3)
assert y_train.shape == (50000, 1)
assert y_test.shape == (10000, 1)

##### Feature Extraction: Canny Edge Detection

In [ ]:
X_train_canny_edge = canny_edge_dataset(X_train)
X_test_canny_edge = canny_edge_dataset(X_test)

Show an example image of the dataset

In [ ]:
img_number = 25

show_image(X_train[img_number])
show_image(X_train_canny_edge[img_number])

##### Feature Extraction: SIFT Transform

In [ ]:
X_train_sift_transform = sift_transform_dataset(X_train)
X_test_sift_transform = sift_transform_dataset(X_test)

In [ ]:
img_number = 1

show_image(X_train[img_number])
show_image(X_train_sift_transform[img_number])

In [ ]:
y_train = y_train.flatten()
y_test = y_test.flatten()

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

We apply one hot encoding to the target labels

In [ ]:
y_train = to_categorical(y_train, num_classes=20)
y_test = to_categorical(y_test, num_classes=20)

In [ ]:
len(X_train), len(y_train), len(X_test), len(y_test)

### Inception

#### Preprocessing

We apply the proper preprocessing to the data for the inception model

In [ ]:
scaled_X_train = keras.applications.inception_v3.preprocess_input(x = X_train_canny_edge)
scaled_X_test = keras.applications.inception_v3.preprocess_input(x = X_test_canny_edge)

#### Creation

We create an inception model, with an upsampling layer to adjust the size of the images to the size required by the model. We opted for stablishing 'max' pooling

In [ ]:
input_tensor = keras.Input(shape=(32, 32, 3))

x = keras.layers.UpSampling2D(size=(3, 3), interpolation='bilinear')(input_tensor)

model_inception = keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    input_tensor=x,
    input_shape=None,
    pooling="max",
    classes=20,
    classifier_activation="softmax",
    name="inception_v3"
)

model_5 = keras.Model(inputs=input_tensor, outputs=model_inception.output)

In [ ]:
visualkeras.layered_view(model_5, legend=True)

#### Compilation

In [ ]:
model_5.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

#### Training

In [ ]:
early_stopping_cb = EarlyStopping(patience=15, monitor="val_loss", mode="min", restore_best_weights=True)

In [ ]:
np.size(y_train)

In [ ]:
history_model_5 = model_5.fit(
    scaled_X_train,
    y_train,
    validation_split=0.2,
    epochs=200,
    callbacks=[early_stopping_cb]
)

#### Testing

As it can be seen in the training output, this model overfits rapidly to the data, but the regularization methods recover the model that was not yet overfitted. 

In [ ]:
model_5.evaluate(scaled_X_test, y_test)

In [ ]:
model_5.save("Model5.h5")

In [ ]:
pd.DataFrame(history_model_5.history).to_csv("Model5_Train_History.csv")
pd.DataFrame(history_model_5.history).plot()

---

### Xception

#### Preprocessing

We apply the proper preprocessing to the data

In [ ]:
scaled_X_train = keras.applications.xception.preprocess_input(x = X_train)
scaled_X_test = keras.applications.xception.preprocess_input(x = X_test)

#### Creation

We upscale the images as we did with previous model

In [ ]:
input_tensor = keras.Input(shape=(32, 32, 3))
x = keras.layers.UpSampling2D(size=(3, 3), interpolation='bilinear')(input_tensor)

model_xception = keras.applications.Xception(
    include_top=True,
    weights=None,
    input_tensor=x,
    input_shape=None,
    pooling="avg",
    classes=20,
    classifier_activation="softmax",
    name="xception"
)

model_6 = keras.Model(inputs=input_tensor, outputs=model_xception.output)

In [ ]:
visualkeras.layered_view(model_6, legend=True)

#### Compilation

In [ ]:
model_6.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

#### Training

In [ ]:
early_stopping_cb = EarlyStopping(patience=15, monitor="val_loss", mode="min", restore_best_weights=True)

In [ ]:
history_model_6 = model_6.fit(
    scaled_X_train,
    y_train,
    validation_split=0.2,
    epochs=200,
    callbacks=[early_stopping_cb]
)

#### Testing

With the Xception model, we obtain our best results, thanks to the model itself, but also to the preprocessing and the regularization methods

In [ ]:
model_6.evaluate(scaled_X_test, y_test)

In [ ]:
model_6.save("Model6.h5")

In [ ]:
pd.DataFrame(history_model_6.history).to_csv("Model6_Train_History.csv")
pd.DataFrame(history_model_6.history).plot()

---

### Residual

#### Preprocessing

We apply again the correct preprocessing

In [ ]:
scaled_X_train = keras.applications.resnet50.preprocess_input(x = X_train)
scaled_X_test = keras.applications.resnet50.preprocess_input(x = X_test)

#### Creation

In [ ]:
input_tensor = keras.Input(shape=(32, 32, 3))
x = keras.layers.UpSampling2D(size=(3, 3), interpolation='bilinear')(input_tensor)

model_residual = tf.keras.applications.ResNet50(
    include_top=True,
    weights=None,
    input_tensor=x,
    input_shape=None,
    pooling='avg',
    classes=20,
    classifier_activation='softmax',
    name='residual'
)

model_7 = keras.Model(inputs=input_tensor, outputs=model_residual.output)

In [ ]:
visualkeras.layered_view(model_7, legend=True)

#### Compilation

In [ ]:
model_7.compile(optimizer='adam',
                loss='categorical_crossentropy',
                metrics=['accuracy'])

#### Training

In [ ]:
early_stopping_cb = EarlyStopping(patience=15, monitor='val_loss', mode='min', restore_best_weights=True)

In [ ]:
history_model_7 = model_7.fit(
    scaled_X_train,
    y_train,
    validation_split=0.2,
    epochs=200,
    callbacks=[early_stopping_cb]
)

#### Testing

The Residual model obtains better results than preivous models except from the Xception model

In [ ]:
model_7.evaluate(scaled_X_test, y_test)

In [ ]:
model_7.save('Model7.h5')

In [ ]:
pd.DataFrame(history_model_7.history).to_csv("Model7_Train_History.csv")
pd.DataFrame(history_model_7.history).plot()